In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

print("hello")

2.11.0+cu128
True
hello


In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [5]:
from src.pannuke_tissue.data import PanNukeTissueDataset

DATA_DIR = Path.home() / "datasets" / "PanNuke" / "fold_1"

IMAGE_PATH = DATA_DIR / "images" / "fold1" / "images.npy"
TYPE_PATH = DATA_DIR / "images" / "fold1" / "types.npy"
# MASK_PATH = DATA_DIR / "masks" / "fold1" / "masks.npy"

dataset = PanNukeTissueDataset(
    IMAGE_PATH,
    TYPE_PATH
)

print("Dataset length:", len(dataset))

image, label = dataset[0]

print(type(image))
print("Min:", image.min())
print("Max:", image.max())
print("Image shape:", image.shape)
print("Image dtype:", image.dtype)
print("Tissue label:", label)

Dataset length: 2656
<class 'torch.Tensor'>
Min: tensor(0.1098)
Max: tensor(1.)
Image shape: torch.Size([3, 256, 256])
Image dtype: torch.float32
Tissue label: 3


In [6]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = PanNukeTissueDataset(
    IMAGE_PATH,
    TYPE_PATH,
    transform=train_transform
)

image, label = dataset[0]

print(type(image))
print("Shape:", image.shape)
print("Dtype:", image.dtype)
print("Min:", image.min())
print("Max:", image.max())
print("Label:", label)

<class 'torch.Tensor'>
Shape: torch.Size([3, 256, 256])
Dtype: torch.float32
Min: tensor(-1.5455)
Max: tensor(2.6400)
Label: 3


In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset, 
    batch_size=32, 
    shuffle=True # reshuffle the training samples each epoch
)

images, labels = next(iter(train_loader)) # get one batch from the DataLoader

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Images dtype:", images.dtype)
print("Labels:", labels[:10])

Images shape: torch.Size([32, 3, 256, 256])
Labels shape: torch.Size([32])
Images dtype: torch.float32
Labels: tensor([ 3,  3,  0, 11,  5,  5,  3,  5,  3,  1])


In [14]:
import torch

from torchvision.models import resnet50, ResNet50_Weights

from src.pannuke_tissue.data import (
    PanNukeTissueDataset,
    TISSUE_CLASSES,
)

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

model.fc = torch.nn.Linear(
    model.fc.in_features,  # how many features enter the final layer / 2048
    len(TISSUE_CLASSES) # output layer predicts the number of tissue classes / 19
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
) # cuda means RTX 3090 GPU or CPU

model = model.to(device) # model -> GPU
images = images.to(device) # images in current batch -> GPU
labels = labels.to(device) # labels in current batch -> GPU

# forward pass through the model
# 32 images [32, 3, 256, 256] -> Resnet-50 backbone -> 2048 features -> final Linear layer -> model's raw prediction scores (= logits) (not probabilities yet) [32, 19]
outputs = model(images)

print("Device:", device)
print("Output shape:", outputs.shape)
print("Output device:", outputs.device)

criterion = torch.nn.CrossEntropyLoss()

loss = criterion(outputs, labels) # the overall loss for that batch of 32 samples

print("Loss:", loss.item())

Device: cuda
Output shape: torch.Size([32, 19])
Output device: cuda:0
Loss: 2.959167957305908


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

optimizer.zero_grad() # to avoid gradient accumulation, this clears the gradients from the previous batch before calculating new gradients for the current batch.

outputs = model(images)

loss = criterion(outputs, labels)
loss.backward() # calculate gradient / in only training set

optimizer.step() # change weights / in only training set

# without calculating gradients, run the same batch through the updated model and calculate its new loss
with torch.no_grad():
    outputs_after = model(images)
    loss_after = criterion(outputs_after, labels)

print("Loss:", loss.item()) 

Loss: 2.508327007293701


For each batch, the loss can go up or down because each batch contains a different mix of images and difficulty levels. What matters is the overall trend across many batches and epochs.  

Some batches are simply harder for the model than others


In [17]:
model.train()

running_loss = 0.0

# training 83 batches = 1 epoch
for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

average_loss = running_loss / len(train_loader)

print("Average training loss:", average_loss)

Average training loss: 1.7820711092776562
